In [20]:
import os
import pandas as pd
import torch.nn as nn
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader,Dataset , random_split
from PIL import Image, ImageFile
from torchvision import transforms , datasets,models

In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [22]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mehmoodsheikh/fairface-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fairface-dataset' dataset.
Path to dataset files: /kaggle/input/fairface-dataset


In [23]:
for file in os.listdir(path):
  print(file)


FairFace


In [24]:
file_path = os.path.join(path,"FairFace/")
print("Contents of the FairFace directory:")
for item in os.listdir(file_path):
  print(item)


Contents of the FairFace directory:
fairface_label_val.csv
fairface_label_train.csv
val
train


In [25]:
face_Data = os.path.join(path,"FairFace","train")
with os.scandir(face_Data) as files:
  for file in files:
    print(file.name)

Streaming output truncated to the last 5000 lines.
77173.jpg
2685.jpg
42732.jpg
71637.jpg
47849.jpg
35889.jpg
50348.jpg
53430.jpg
70550.jpg
60443.jpg
9058.jpg
46318.jpg
61124.jpg
68128.jpg
6153.jpg
39372.jpg
679.jpg
22600.jpg
57772.jpg
23315.jpg
71244.jpg
65957.jpg
74586.jpg
67995.jpg
53404.jpg
72768.jpg
38894.jpg
86316.jpg
70650.jpg
45111.jpg
7524.jpg
9278.jpg
64578.jpg
68564.jpg
13063.jpg
73079.jpg
1264.jpg
37180.jpg
20296.jpg
75698.jpg
26737.jpg
44513.jpg
71818.jpg
63622.jpg
13481.jpg
81677.jpg
65137.jpg
65695.jpg
22476.jpg
46555.jpg
31310.jpg
62194.jpg
53238.jpg
42477.jpg
1869.jpg
73405.jpg
4579.jpg
45954.jpg
44093.jpg
36908.jpg
24696.jpg
9807.jpg
86642.jpg
24342.jpg
39625.jpg
46675.jpg
21184.jpg
17210.jpg
56594.jpg
57215.jpg
27316.jpg
3416.jpg
44332.jpg
25653.jpg
4636.jpg
25656.jpg
2757.jpg
21669.jpg
26278.jpg
11035.jpg
31856.jpg
11507.jpg
26585.jpg
80385.jpg
714.jpg
57953.jpg
33349.jpg
38252.jpg
3589.jpg
42082.jpg
32797.jpg
542.jpg
12309.jpg
59568.jpg
75605.jpg
65166.jpg
13154.jp

In [26]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

In [56]:
ImageFile.LOAD_TRUNCATED_IMAGES = True
def pil_loader_robust(path):
    try:
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('RGB')
    except OSError as e:
        print(f"Error loading image {path}: {e}")
        return Image.new('RGB', (128, 128)) # Return a blank image or handle as appropriate

class FairFaceCustomDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None, loader=pil_loader_robust, label_column='race'):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform
        self.loader = loader
        self.label_column = label_column # Store the label column name - FIX applied here
        # Map labels to integers and store classes based on the specified label_column
        self.classes = sorted(self.dataframe[self.label_column].unique())
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx, 0]) # Assuming file name is the first column
        image = self.loader(img_name)

        # Get the label from the specified column and convert to integer
        label_value = self.dataframe.iloc[idx][self.label_column]
        label = self.class_to_idx[label_value]

        if self.transform:
            image = self.transform(image)
        return image, label

# Load the training DataFrame (assuming df from AlW8aOH9284D is available in scope)
train_df = pd.read_csv(os.path.join(path, "FairFace", "fairface_label_train.csv"))
train_images_root = os.path.join(path, "FairFace") # Corrected path to point to the parent of 'train' folder

# Create the dataset (defaulting to 'race' if not specified, for backward compatibility)
dataset = FairFaceCustomDataset(
    dataframe=train_df,
    root_dir=train_images_root,
    transform=transform,
    loader=pil_loader_robust,
    label_column='race' # Explicitly set default for existing 'dataset' object
)

### Age Classification Setup

In [57]:
# Load the training DataFrame
train_df = pd.read_csv(os.path.join(path, "FairFace", "fairface_label_train.csv"))
train_images_root = os.path.join(path, "FairFace")

# Create the dataset for Age classification
dataset_age = FairFaceCustomDataset(
    dataframe=train_df,
    root_dir=train_images_root,
    transform=transform,
    loader=pil_loader_robust,
    label_column='age' # Specify 'age' as the label column
)

print(f"Number of age classes: {len(dataset_age.classes)}")
print(f"Age classes: {dataset_age.classes}")

Number of age classes: 9
Age classes: ['0-2', '10-19', '20-29', '3-9', '30-39', '40-49', '50-59', '60-69', 'more than 70']


In [58]:
# Split the age dataset into training and testing sets
Train_dataset_age, Test_dataset_age = random_split(
    dataset_age,
    [0.8, 0.2]
)

# Explicitly set the robust loader for the underlying datasets of the subsets
# This might be redundant if the parent dataset's loader is already robust, but ensures it.
# Note: random_split wraps the original dataset, so we need to access its .dataset attribute
if isinstance(Train_dataset_age.dataset, FairFaceCustomDataset):
    Train_dataset_age.dataset.loader = pil_loader_robust
if isinstance(Test_dataset_age.dataset, FairFaceCustomDataset):
    Test_dataset_age.dataset.loader = pil_loader_robust

print(f"Length of age training dataset: {len(Train_dataset_age)}")
print(f"Length of age testing dataset: {len(Test_dataset_age)}")

Length of age training dataset: 69396
Length of age testing dataset: 17348


In [59]:
# Create DataLoaders for Age classification
train_loader_age = DataLoader(
    dataset=Train_dataset_age,
    batch_size=32,
    shuffle=True
)

test_loader_age = DataLoader(
    dataset=Test_dataset_age,
    batch_size=32,
    shuffle=True
)

# Verify a batch from the age loader
images_age, labels_age = next(iter(train_loader_age))
print(f"Batch image shape (Age): {images_age.shape}")
print(f"Batch labels (Age): {labels_age}")

Batch image shape (Age): torch.Size([32, 3, 128, 128])
Batch labels (Age): tensor([2, 5, 1, 2, 7, 1, 2, 2, 8, 4, 3, 2, 2, 4, 2, 6, 4, 2, 2, 5, 1, 1, 4, 5,
        2, 4, 2, 6, 4, 6, 0, 5])


### Gender Classification Setup

In [60]:
# Create the dataset for Gender classification
dataset_gender = FairFaceCustomDataset(
    dataframe=train_df,
    root_dir=train_images_root,
    transform=transform,
    loader=pil_loader_robust,
    label_column='gender' # Specify 'gender' as the label column
)

print(f"Number of gender classes: {len(dataset_gender.classes)}")
print(f"Gender classes: {dataset_gender.classes}")

Number of gender classes: 2
Gender classes: ['Female', 'Male']


In [61]:
# Split the gender dataset into training and testing sets
Train_dataset_gender, Test_dataset_gender = random_split(
    dataset_gender,
    [0.8, 0.2]
)

# Explicitly set the robust loader for the underlying datasets of the subsets
if isinstance(Train_dataset_gender.dataset, FairFaceCustomDataset):
    Train_dataset_gender.dataset.loader = pil_loader_robust
if isinstance(Test_dataset_gender.dataset, FairFaceCustomDataset):
    Test_dataset_gender.dataset.loader = pil_loader_robust

print(f"Length of gender training dataset: {len(Train_dataset_gender)}")
print(f"Length of gender testing dataset: {len(Test_dataset_gender)}")

Length of gender training dataset: 69396
Length of gender testing dataset: 17348


In [62]:
# Create DataLoaders for Gender classification
train_loader_gender = DataLoader(
    dataset=Train_dataset_gender,
    batch_size=32,
    shuffle=True
)

test_loader_gender = DataLoader(
    dataset=Test_dataset_gender,
    batch_size=32,
    shuffle=True
)

# Verify a batch from the gender loader
images_gender, labels_gender = next(iter(train_loader_gender))
print(f"Batch image shape (Gender): {images_gender.shape}")
print(f"Batch labels (Gender): {labels_gender}")

Batch image shape (Gender): torch.Size([32, 3, 128, 128])
Batch labels (Gender): tensor([1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0,
        0, 1, 0, 1, 0, 0, 0, 1])


### Age Model Training

In [63]:
class CNN_Age(nn.Module):
  def __init__(self, num_classes):
    super(CNN_Age,self).__init__()
    self.conv1 = nn.Conv2d(3,16,3,padding=1)
    self.conv2 = nn.Conv2d(16,32,3,padding=1)
    self.conv3 = nn.Conv2d(32,64,3,padding=1)
    self.pool = nn.MaxPool2d(2,2)
    self.fc1 = nn.Linear(64*16*16,512)
    self.fc2 = nn.Linear(512,256)
    self.fc3 = nn.Linear(256,num_classes) # Dynamic output features

  def forward(self,x):
    x = self.pool(nn.functional.relu(self.conv1(x)))
    x = self.pool(nn.functional.relu(self.conv2(x)))
    x = self.pool(nn.functional.relu(self.conv3(x)))
    x = x.view(-1,64*16*16)
    x = nn.functional.relu(self.fc1(x))
    x = nn.functional.relu(self.fc2(x))
    x = self.fc3(x)
    return x

model_age = CNN_Age(num_classes=len(dataset_age.classes))
model_age.to(device)
print(model_age)

optimizer_age = Adam(model_age.parameters(),lr=0.001)
criterion_age = nn.CrossEntropyLoss()

CNN_Age(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=16384, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=9, bias=True)
)


In [64]:
epochs = 10
for epoch in range(epochs):
  model_age.train()

  running_loss = 0
  correct = 0
  total = 0

  for images, labels in train_loader_age:
    images = images.to(device)
    labels = labels.to(device)
    optimizer_age.zero_grad() # Use optimizer_age
    outputs = model_age(images) # Use model_age
    loss = criterion_age(outputs, labels) # Use criterion_age
    loss.backward()
    optimizer_age.step() # Use optimizer_age
    running_loss += loss.item()
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
  train_loss = running_loss / len(train_loader_age)
  train_acc = 100 * correct / total
  print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Age Train Loss: {train_loss:.4f} "
        f"Age Train Accuracy: {train_acc:.2f}%"
  )

# Evaluate Age Model
model_age.eval() # Ensure model is on the correct device
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader_age:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model_age(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
test_accuracy_age = 100 * correct / total
print(f"\nAge Test Accuracy : {test_accuracy_age:.2f}%")
torch.save(
    model_age.state_dict(),
    "age_model.pth"
  )

Epoch [1/10] Age Train Loss: 1.6258 Age Train Accuracy: 36.04%
Epoch [2/10] Age Train Loss: 1.3823 Age Train Accuracy: 43.47%
Epoch [3/10] Age Train Loss: 1.2720 Age Train Accuracy: 47.26%
Epoch [4/10] Age Train Loss: 1.1757 Age Train Accuracy: 50.72%
Epoch [5/10] Age Train Loss: 1.0784 Age Train Accuracy: 54.45%
Epoch [6/10] Age Train Loss: 0.9674 Age Train Accuracy: 59.33%
Epoch [7/10] Age Train Loss: 0.8437 Age Train Accuracy: 64.30%
Epoch [8/10] Age Train Loss: 0.7146 Age Train Accuracy: 70.02%
Epoch [9/10] Age Train Loss: 0.5972 Age Train Accuracy: 75.17%
Epoch [10/10] Age Train Loss: 0.5015 Age Train Accuracy: 79.59%

Age Test Accuracy : 44.74%


### Gender Model Training

In [65]:
class CNN_Gender(nn.Module):
  def __init__(self, num_classes):
    super(CNN_Gender,self).__init__()
    self.conv1 = nn.Conv2d(3,16,3,padding=1)
    self.conv2 = nn.Conv2d(16,32,3,padding=1)
    self.conv3 = nn.Conv2d(32,64,3,padding=1)
    self.pool = nn.MaxPool2d(2,2)
    self.fc1 = nn.Linear(64*16*16,512)
    self.fc2 = nn.Linear(512,256)
    self.fc3 = nn.Linear(256,num_classes) # Dynamic output features

  def forward(self,x):
    x = self.pool(nn.functional.relu(self.conv1(x)))
    x = self.pool(nn.functional.relu(self.conv2(x)))
    x = self.pool(nn.functional.relu(self.conv3(x)))
    x = x.view(-1,64*16*16)
    x = nn.functional.relu(self.fc1(x))
    x = nn.functional.relu(self.fc2(x))
    x = self.fc3(x)
    return x

model_gender = CNN_Gender(num_classes=len(dataset_gender.classes))
model_gender.to(device)
print(model_gender)

optimizer_gender = Adam(model_gender.parameters(),lr=0.001)
criterion_gender = nn.CrossEntropyLoss()

CNN_Gender(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=16384, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=2, bias=True)
)


In [66]:
epochs = 10
for epoch in range(epochs):
  model_gender.train()

  running_loss = 0
  correct = 0
  total = 0

  for images, labels in train_loader_gender:
    images = images.to(device)
    labels = labels.to(device)
    optimizer_gender.zero_grad() # Use optimizer_gender
    outputs = model_gender(images) # Use model_gender
    loss = criterion_gender(outputs, labels) # Use criterion_gender
    loss.backward()
    optimizer_gender.step() # Use optimizer_gender
    running_loss += loss.item()
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
  train_loss = running_loss / len(train_loader_gender)
  train_acc = 100 * correct / total
  print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Gender Train Loss: {train_loss:.4f} "
        f"Gender Train Accuracy: {train_acc:.2f}%"
  )

# Evaluate Gender Model
model_gender.eval() # Ensure model is on the correct device
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader_gender:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model_gender(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
test_accuracy_gender = 100 * correct / total
print(f"\nGender Test Accuracy : {test_accuracy_gender:.2}%")
torch.save(
    model_gender.state_dict(),
    "gender_model.pth"
  )

Epoch [1/10] Gender Train Loss: 0.5303 Gender Train Accuracy: 71.64%
Epoch [2/10] Gender Train Loss: 0.4231 Gender Train Accuracy: 79.23%
Epoch [3/10] Gender Train Loss: 0.3760 Gender Train Accuracy: 82.00%
Epoch [4/10] Gender Train Loss: 0.3377 Gender Train Accuracy: 84.15%
Epoch [5/10] Gender Train Loss: 0.2990 Gender Train Accuracy: 86.12%
Epoch [6/10] Gender Train Loss: 0.2588 Gender Train Accuracy: 88.31%
Epoch [7/10] Gender Train Loss: 0.2212 Gender Train Accuracy: 90.18%
Epoch [8/10] Gender Train Loss: 0.1845 Gender Train Accuracy: 91.91%
Epoch [9/10] Gender Train Loss: 0.1513 Gender Train Accuracy: 93.59%
Epoch [10/10] Gender Train Loss: 0.1247 Gender Train Accuracy: 94.74%

Gender Test Accuracy : 8.1e+01%


### Updated Prediction Function and Gradio Interface

In [67]:
# Load the trained models
model_age_loaded = CNN_Age(num_classes=len(dataset_age.classes))
model_age_loaded.load_state_dict(torch.load("age_model.pth"))
model_age_loaded.to(device)
model_age_loaded.eval()

model_gender_loaded = CNN_Gender(num_classes=len(dataset_gender.classes))
model_gender_loaded.load_state_dict(torch.load("gender_model.pth"))
model_gender_loaded.to(device)
model_gender_loaded.eval()

def predict_age_gender(image, task):
  image_tensor = transform(image)
  image_tensor = image_tensor.unsqueeze(0)
  image_tensor = image_tensor.to(device)

  if task == 'Age':
      model = model_age_loaded
      classes = dataset_age.classes
  elif task == 'Gender':
      model = model_gender_loaded
      classes = dataset_gender.classes
  else:
      return {"Error": "Invalid task selected"}

  with torch.no_grad():
      outputs = model(image_tensor)
      probabilities = torch.softmax(outputs, dim=1)[0]

  top_values, top_indices = torch.topk(probabilities, min(5, len(classes)))
  result = {}

  for value, index in zip(top_values, top_indices):
      class_name = classes[index.item()]
      result[class_name] = float(value.item())

  print(f"Prediction for {task}:", result)
  return result


In [68]:
import gradio as gr

if 'demo_age_gender' in locals(): # Check if demo already exists
    demo_age_gender.close() # Close existing demo if running

demo_age_gender = gr.Interface(
    fn=predict_age_gender,
    inputs=[
        gr.Image(
            type="pil",
            label="Upload Human Image "
        ),
        gr.Radio(
            choices=['Age', 'Gender'],
            label="Select Task",
            value='Age' # Default selection
        )
    ],
    outputs=gr.Label(
        num_top_classes=5,
        label="Prediction"
    ),
    title="Human Age and Gender Classification",
    description=(
        "Upload a human image to predict either their age or gender."
    )
)

if __name__ == "__main__":
    demo_age_gender.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://951420736d18051a70.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
